# 🏥 Hospital Patient Care Analytics Pipeline

A beginner-friendly Data Engineering project that extracts hospital data, cleans it, validates it, stores it in MySQL, and performs basic SQL analytics.

*A Basic Data Engineering / ETL project built with Python, Pandas, SQLAlchemy and MySQL.*


## 🎯 Project Objective

A multi-specialty hospital collects data from many different places: the front desk where patients register, the appointment desk, the laboratory, wearable health devices patients use at home, and the notes doctors write after a consultation.

**Why does this matter?** Each of these systems stores data in its own way, with its own small mistakes - a missing value here, a duplicate entry there, a date written in two different styles. If the hospital wants to actually *understand* its patients (how busy each department is, what the average patient looks like, which patients might need extra attention), it first needs all this scattered, messy data brought together, cleaned up, and stored somewhere it can be safely queried.

**What this project does, in one sentence:** it reads the raw hospital data, cleans and checks it, stores it in a MySQL database, and then runs simple SQL queries on top of it - the basic building blocks of Data Engineering.

We are **not** building a Machine Learning model here. This notebook is only about getting the data ready. A model could be trained later using exactly the clean data this notebook produces.


## 📂 Data Sources

The hospital has five sources of data, and each one becomes one table in our pipeline:

1. **Patient Registration** - basic information collected when a patient signs up
2. **Appointment Scheduling** - which doctor a patient is seeing, and when
3. **Laboratory Reports** - test results ordered for a patient
4. **Wearable Health Data** - heart rate, steps and sleep collected by a wearable device
5. **Doctor Consultation Notes** - what the doctor observed and recommended during a visit

```text
Patient Registration → patients
Appointments          → appointments
Laboratory Reports    → lab_reports
Wearable Devices       → wearable_data
Doctor Notes            → doctor_notes
```


## 🧾 Creating the Dataset

Since this is a demo project, there isn't a real hospital system to pull data from. So in this section we create **fictional** hospital data ourselves, directly inside the notebook using Pandas, and save it as CSV files - exactly like the files a real hospital's systems would export.

A few realistic data-quality problems are included **on purpose** in this data - a missing value, a duplicate row, an inconsistent format, and a value that needs converting. This gives our Transform step something real to actually clean, instead of working on data that's already perfect.

All names, IDs, and values below are made up and do not represent any real person.


In [39]:
import pandas as pd
import os

os.makedirs("data", exist_ok=True)
pd.set_option("display.max_columns", None)


In [40]:
# ---------- 1. Patients ----------
# A missing age (HP008), a duplicate row (HP001 appears twice),
# an inconsistent gender format (Male / male / M / F / female),
# and a registration date written in a different style for one row (04-04-2024).

patients_rows = [
    ["HP001", "Nikhil Bhat", 29, "Male", "9123456780", "2024-04-01"],
    ["HP002", "Ayesha Khan", 34, "Female", "9123456781", "2024-04-02"],
    ["HP003", "Rohan Deshmukh", 41, "male", "9123456782", "2024-04-03"],
    ["HP004", "Kavya Suresh", 26, "F", "9123456783", "2024-04-04"],
    ["HP005", "Manoj Tiwari", 50, "M", "9123456784", "04-04-2024"],
    ["HP006", "Zara Ahmed", 31, "female", "9123456785", "2024-04-05"],
    ["HP007", "Aditya Rane", 38, "Male", "9123456786", "2024-04-06"],
    ["HP008", "Tanvi Joshi", None, "Female", "9123456787", "2024-04-07"],
    ["HP009", "Farhan Sheikh", 55, "male", "9123456788", "2024-04-08"],
    ["HP010", "Isha Kulkarni", 22, "F", "9123456789", "2024-04-09"],
    ["HP011", "Devansh Rathi", 44, "M", "9123456790", "2024-04-10"],
    ["HP012", "Riya Chawla", 27, "Female", "9123456791", "2024-04-11"],
    ["HP013", "Kabir Malhotra", 58, "male", "9123456792", "2024-04-12"],
    ["HP014", "Simran Kaur", 30, "F", "9123456793", "2024-04-13"],
    ["HP015", "Yash Patil", 35, "Male", "9123456794", "2024-04-14"],
    ["HP016", "Meher Fernandes", 24, "female", "9123456795", "2024-04-15"],
    ["HP017", "Aarav Nanda", 47, "M", "9123456796", "2024-04-16"],
    ["HP018", "Divya Prasad", 39, "Female", "9123456797", "2024-04-17"],
    ["HP001", "Nikhil Bhat", 29, "Male", "9123456780", "2024-04-01"],  # duplicate row
]
patients = pd.DataFrame(
    patients_rows,
    columns=["patient_id", "name", "age", "gender", "phone", "registration_date"],
)
patients.to_csv("data/patients.csv", index=False)
patients.head()


,patient_id,name,age,gender,phone,registration_date
0,HP001,Nikhil Bhat,29.0,Male,9123456780,2024-04-01
1,HP002,Ayesha Khan,34.0,Female,9123456781,2024-04-02
2,HP003,Rohan Deshmukh,41.0,male,9123456782,2024-04-03
3,HP004,Kavya Suresh,26.0,F,9123456783,2024-04-04
4,HP005,Manoj Tiwari,50.0,M,9123456784,04-04-2024


In [41]:
# ---------- 2. Appointments ----------
# Two missing status values, and one duplicate row (AP001 appears twice).

appointment_rows = [
    ["AP001", "HP001", "DR01", "2024-04-20", "Cardiology", "Completed"],
    ["AP002", "HP002", "DR02", "2024-04-20", "Neurology", "Completed"],
    ["AP003", "HP003", "DR03", "2024-04-21", "Pediatrics", ""],
    ["AP004", "HP004", "DR04", "2024-04-21", "Orthopedics", "Completed"],
    ["AP005", "HP005", "DR02", "2024-04-22", "Neurology", "Cancelled"],
    ["AP006", "HP006", "DR05", "2024-04-22", "Gynecology", "Completed"],
    ["AP007", "HP007", "DR01", "2024-04-23", "Cardiology", "Scheduled"],
    ["AP008", "HP008", "DR06", "2024-04-23", "General Medicine", "Completed"],
    ["AP009", "HP009", "DR04", "2024-04-24", "Orthopedics", "Completed"],
    ["AP010", "HP010", "DR02", "2024-04-24", "Neurology", ""],
    ["AP011", "HP011", "DR05", "2024-04-25", "Gynecology", "Completed"],
    ["AP012", "HP012", "DR01", "2024-04-25", "Cardiology", "Completed"],
    ["AP013", "HP013", "DR06", "2024-04-26", "General Medicine", "Scheduled"],
    ["AP014", "HP014", "DR04", "2024-04-26", "Orthopedics", "Completed"],
    ["AP015", "HP015", "DR02", "2024-04-27", "Neurology", "Completed"],
    ["AP016", "HP016", "DR05", "2024-04-27", "Gynecology", "Cancelled"],
    ["AP017", "HP017", "DR01", "2024-04-28", "Cardiology", "Completed"],
    ["AP018", "HP018", "DR06", "2024-04-28", "General Medicine", "Completed"],
    ["AP001", "HP001", "DR01", "2024-04-20", "Cardiology", "Completed"],  # duplicate row
]
appointments = pd.DataFrame(
    appointment_rows,
    columns=["appointment_id", "patient_id", "doctor_id", "appointment_date", "department", "status"],
)
appointments.to_csv("data/appointments.csv", index=False)
appointments.head()


,appointment_id,patient_id,doctor_id,appointment_date,department,status
0,AP001,HP001,DR01,2024-04-20,Cardiology,Completed
1,AP002,HP002,DR02,2024-04-20,Neurology,Completed
2,AP003,HP003,DR03,2024-04-21,Pediatrics,
3,AP004,HP004,DR04,2024-04-21,Orthopedics,Completed
4,AP005,HP005,DR02,2024-04-22,Neurology,Cancelled


In [42]:
# ---------- 3. Lab reports ----------
# A missing test value (LB003), and one value stored as text instead of a number ("not done").

lab_rows = [
    ["LB001", "HP001", "Blood Pressure", "118", "mmHg", "2024-04-20"],
    ["LB002", "HP002", "Vitamin D", "22.5", "ng/mL", "2024-04-20"],
    ["LB003", "HP003", "Thyroid Level", "", "uIU/mL", "2024-04-21"],
    ["LB004", "HP004", "Glucose", "96", "mg/dL", "2024-04-21"],
    ["LB005", "HP005", "Blood Pressure", "not done", "mmHg", "2024-04-22"],
    ["LB006", "HP006", "Vitamin D", "18.2", "ng/mL", "2024-04-22"],
    ["LB007", "HP007", "Glucose", "101", "mg/dL", "2024-04-23"],
    ["LB008", "HP008", "Thyroid Level", "3.1", "uIU/mL", "2024-04-23"],
    ["LB009", "HP009", "Blood Pressure", "124", "mmHg", "2024-04-24"],
    ["LB010", "HP010", "Vitamin D", "25.0", "ng/mL", "2024-04-24"],
    ["LB011", "HP011", "Glucose", "108", "mg/dL", "2024-04-25"],
    ["LB012", "HP012", "Thyroid Level", "2.8", "uIU/mL", "2024-04-25"],
    ["LB013", "HP013", "Blood Pressure", "130", "mmHg", "2024-04-26"],
    ["LB014", "HP014", "Vitamin D", "19.7", "ng/mL", "2024-04-26"],
    ["LB015", "HP015", "Glucose", "93", "mg/dL", "2024-04-27"],
    ["LB016", "HP016", "Thyroid Level", "3.4", "uIU/mL", "2024-04-27"],
    ["LB017", "HP017", "Blood Pressure", "121", "mmHg", "2024-04-28"],
    ["LB018", "HP018", "Vitamin D", "21.1", "ng/mL", "2024-04-28"],
]
lab_reports = pd.DataFrame(
    lab_rows,
    columns=["lab_id", "patient_id", "test_name", "test_value", "test_unit", "test_date"],
)
lab_reports.to_csv("data/lab_reports.csv", index=False)
lab_reports.head()


,lab_id,patient_id,test_name,test_value,test_unit,test_date
0,LB001,HP001,Blood Pressure,118,mmHg,2024-04-20
1,LB002,HP002,Vitamin D,22.5,ng/mL,2024-04-20
2,LB003,HP003,Thyroid Level,,uIU/mL,2024-04-21
3,LB004,HP004,Glucose,96,mg/dL,2024-04-21
4,LB005,HP005,Blood Pressure,not done,mmHg,2024-04-22


In [43]:
# ---------- 4. Wearable data ----------
# A missing heart rate (WR003), and a negative step count (WR004) which is physically impossible.

wearable_rows = [
    ["WR001", "HP001", "2024-04-20", 74, 6100, 7.1],
    ["WR002", "HP002", "2024-04-20", 81, 5300, 6.4],
    ["WR003", "HP003", "2024-04-21", None, 7200, 7.6],
    ["WR004", "HP004", "2024-04-21", 76, -400, 6.2],
    ["WR005", "HP005", "2024-04-22", 67, 3100, 8.1],
    ["WR006", "HP006", "2024-04-22", 92, 8300, 5.3],
    ["WR007", "HP007", "2024-04-23", 78, 4700, 7.3],
    ["WR008", "HP008", "2024-04-23", 73, 6000, 6.9],
    ["WR009", "HP009", "2024-04-24", 88, 2200, 7.0],
    ["WR010", "HP010", "2024-04-24", 70, 6800, 6.1],
    ["WR011", "HP011", "2024-04-25", 77, 5400, 7.2],
    ["WR012", "HP012", "2024-04-25", 84, 4800, 6.8],
    ["WR013", "HP013", "2024-04-26", 69, 3800, 7.5],
    ["WR014", "HP014", "2024-04-26", 75, 6200, 6.7],
    ["WR015", "HP015", "2024-04-27", 79, 5600, 7.4],
    ["WR016", "HP016", "2024-04-27", 72, 4500, 6.6],
    ["WR017", "HP017", "2024-04-28", 86, 5100, 7.0],
    ["WR018", "HP018", "2024-04-28", 80, 5900, 6.9],
]
wearable_data = pd.DataFrame(
    wearable_rows,
    columns=["record_id", "patient_id", "record_date", "heart_rate", "steps", "sleep_hours"],
)
wearable_data.to_csv("data/wearable_data.csv", index=False)
wearable_data.head()


,record_id,patient_id,record_date,heart_rate,steps,sleep_hours
0,WR001,HP001,2024-04-20,74.0,6100,7.1
1,WR002,HP002,2024-04-20,81.0,5300,6.4
2,WR003,HP003,2024-04-21,NaN,7200,7.6
3,WR004,HP004,2024-04-21,76.0,-400,6.2
4,WR005,HP005,2024-04-22,67.0,3100,8.1


In [44]:
# ---------- 5. Doctor notes ----------
# A missing diagnosis (DN003), and a duplicate row (DN001 appears twice).

notes_rows = [
    ["DN001", "HP001", "DR01", "Mild Hypertension", "Advised regular BP monitoring", "2024-04-20"],
    ["DN002", "HP002", "DR02", "Migraine", "Prescribed pain relief and rest", "2024-04-20"],
    ["DN003", "HP003", "DR03", "", "Patient reported fatigue and low appetite", "2024-04-21"],
    ["DN004", "HP004", "DR04", "Knee Strain", "Recommended physiotherapy", "2024-04-21"],
    ["DN005", "HP005", "DR02", "Tension Headache", "Suggested stress management techniques", "2024-04-22"],
    ["DN006", "HP006", "DR05", "Routine Checkup", "No concerns found during examination", "2024-04-22"],
    ["DN007", "HP007", "DR01", "Chest Discomfort", "Referred for further cardiac evaluation", "2024-04-23"],
    ["DN008", "HP008", "DR06", "Seasonal Flu", "Advised rest, fluids and paracetamol", "2024-04-23"],
    ["DN009", "HP009", "DR04", "Lower Back Pain", "Recommended stretching exercises", "2024-04-24"],
    ["DN010", "HP010", "DR02", "Vertigo", "Prescribed medication and follow-up", "2024-04-24"],
    ["DN011", "HP011", "DR05", "Routine Checkup", "No concerns found during examination", "2024-04-25"],
    ["DN012", "HP012", "DR01", "Palpitations", "Advised ECG and lifestyle changes", "2024-04-25"],
    ["DN013", "HP013", "DR06", "Common Cold", "Advised rest and hydration", "2024-04-26"],
    ["DN014", "HP014", "DR04", "Joint Stiffness", "Recommended light exercise routine", "2024-04-26"],
    ["DN015", "HP015", "DR02", "Mild Migraine", "Prescribed pain relief medication", "2024-04-27"],
    ["DN016", "HP016", "DR05", "Prenatal Checkup", "Routine prenatal examination completed", "2024-04-27"],
    ["DN017", "HP017", "DR01", "Mild Hypertension", "Advised regular BP monitoring", "2024-04-28"],
    ["DN018", "HP018", "DR06", "Viral Fever", "Advised rest, fluids and monitoring", "2024-04-28"],
    ["DN001", "HP001", "DR01", "Mild Hypertension", "Advised regular BP monitoring", "2024-04-20"],  # duplicate row
]
doctor_notes = pd.DataFrame(
    notes_rows,
    columns=["note_id", "patient_id", "doctor_id", "diagnosis", "notes", "visit_date"],
)
doctor_notes.to_csv("data/doctor_notes.csv", index=False)
doctor_notes.head()


,note_id,patient_id,doctor_id,diagnosis,notes,visit_date
0,DN001,HP001,DR01,Mild Hypertension,Advised regular BP monitoring,2024-04-20
1,DN002,HP002,DR02,Migraine,Prescribed pain relief and rest,2024-04-20
2,DN003,HP003,DR03,,Patient reported fatigue and low appetite,2024-04-21
3,DN004,HP004,DR04,Knee Strain,Recommended physiotherapy,2024-04-21
4,DN005,HP005,DR02,Tension Headache,Suggested stress management techniques,2024-04-22


## 📥 Step 1: Extract

Extract means reading the raw hospital data from the CSV files and bringing it into Python so we can work with it. We just saved five CSV files above - now we read them back in, exactly like we would if these files had come from the hospital's own systems.


In [45]:
patients = pd.read_csv("data/patients.csv")
appointments = pd.read_csv("data/appointments.csv")
lab_reports = pd.read_csv("data/lab_reports.csv")
wearable_data = pd.read_csv("data/wearable_data.csv")
doctor_notes = pd.read_csv("data/doctor_notes.csv")

print("=" * 40)
print("          EXTRACT STEP")
print("=" * 40)
print(f"Patients       : {len(patients)} rows")
print(f"Appointments   : {len(appointments)} rows")
print(f"Lab Reports    : {len(lab_reports)} rows")
print(f"Wearable Data  : {len(wearable_data)} rows")
print(f"Doctor Notes   : {len(doctor_notes)} rows")
print("=" * 40)


          EXTRACT STEP
Patients       : 19 rows
Appointments   : 19 rows
Lab Reports    : 18 rows
Wearable Data  : 18 rows
Doctor Notes   : 19 rows


## 🔍 Step 2: Understand the Raw Data

Before cleaning the data, we first check what problems exist. This helps us know exactly what needs to be fixed, instead of guessing. For each dataset we look at:

- **Shape** - how many rows and columns it has
- **Duplicates** - rows that are exact repeats of another row
- **Missing values** - blanks that should have a value

This is a quick health check on the raw data, done before we touch anything.


In [46]:
def inspect(name, df):
    print(f"{name} Dataset")
    print("-" * len(f"{name} Dataset"))
    print(f"Rows       : {df.shape[0]}")
    print(f"Columns    : {df.shape[1]}")
    print(f"Duplicates : {df.duplicated().sum()}")
    print(f"Missing values: {df.isna().sum().sum()}")
    print()


inspect("Patients", patients)
inspect("Appointments", appointments)
inspect("Lab Reports", lab_reports)
inspect("Wearable Data", wearable_data)
inspect("Doctor Notes", doctor_notes)


Patients Dataset
----------------
Rows       : 19
Columns    : 6
Duplicates : 1
Missing values: 1

Appointments Dataset
--------------------
Rows       : 19
Columns    : 6
Duplicates : 1
Missing values: 2

Lab Reports Dataset
-------------------
Rows       : 18
Columns    : 6
Duplicates : 0
Missing values: 1

Wearable Data Dataset
---------------------
Rows       : 18
Columns    : 6
Duplicates : 0
Missing values: 1

Doctor Notes Dataset
--------------------
Rows       : 19
Columns    : 6
Duplicates : 1
Missing values: 1



## 🧹 Step 3: Transform

Transform means cleaning the raw data so that it becomes consistent and easier to use.

For each problem we found above, we follow the same simple pattern:

**Problem → What we do → Why**

We clean each of the five datasets one at a time below.


In [47]:
def parse_date_column(series):
    """
    Some of our date columns mix two different styles
    (e.g. "2024-04-04" and "04-04-2024"). This helper tries each known
    format in turn, so every row gets read correctly instead of pandas
    guessing one single format for the whole column.
    """
    known_formats = ["%Y-%m-%d", "%d-%m-%Y"]
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in known_formats:
        still_missing = result.isna()
        parsed = pd.to_datetime(series[still_missing], format=fmt, errors="coerce")
        result.loc[still_missing] = parsed
    return result


In [48]:
# ---------- Clean patients ----------
# Problem: an exact duplicate patient row.
# What we do: drop duplicate rows.
# Why: a duplicate patient would make our patient count wrong.
patients_clean = patients.drop_duplicates().copy()

# Problem: gender is written inconsistently ("Male", "male", "M", "F", "Female", "female").
# What we do: map everything to just "Male" or "Female".
# Why: if "Male" and "M" are treated as different values, grouping by gender gives wrong results.
gender_map = {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
patients_clean["gender"] = (
    patients_clean["gender"].astype(str).str.strip().str.lower().map(gender_map)
)

# Problem: one patient is missing an age.
# What we do: fill it with the median (middle) age of all patients.
# Why: this is a safe, simple estimate that won't skew the data much.
median_age = patients_clean["age"].median()
patients_clean["age"] = patients_clean["age"].fillna(median_age).astype(int)

# Problem: registration dates are written in two different styles.
# What we do: convert the column into one real, consistent date type.
# Why: real dates let SQL and Python sort and filter correctly.
patients_clean["registration_date"] = parse_date_column(patients_clean["registration_date"])
patients_clean["phone"] = patients_clean["phone"].astype(str).str.strip()

patients_clean.head()


,patient_id,name,age,gender,phone,registration_date
0,HP001,Nikhil Bhat,29,Male,9123456780,2024-04-01
1,HP002,Ayesha Khan,34,Female,9123456781,2024-04-02
2,HP003,Rohan Deshmukh,41,Male,9123456782,2024-04-03
3,HP004,Kavya Suresh,26,Female,9123456783,2024-04-04
4,HP005,Manoj Tiwari,50,Male,9123456784,2024-04-04


In [49]:
# ---------- Clean appointments ----------
# Problem: an exact duplicate appointment row.
# What we do: drop duplicate rows.
appointments_clean = appointments.drop_duplicates().copy()

# Problem: a few appointments have no status.
# What we do: fill missing status with "Unknown" instead of leaving it blank.
# Why: a blank value can break grouping and cause errors later, "Unknown" is an honest label.
appointments_clean["status"] = appointments_clean["status"].fillna("Unknown")
appointments_clean["status"] = appointments_clean["status"].replace("", "Unknown")

# Problem: appointment dates are stored as plain text.
# What we do: convert to a real date type.
appointments_clean["appointment_date"] = pd.to_datetime(
    appointments_clean["appointment_date"], errors="coerce"
)

appointments_clean.head()


,appointment_id,patient_id,doctor_id,appointment_date,department,status
0,AP001,HP001,DR01,2024-04-20,Cardiology,Completed
1,AP002,HP002,DR02,2024-04-20,Neurology,Completed
2,AP003,HP003,DR03,2024-04-21,Pediatrics,Unknown
3,AP004,HP004,DR04,2024-04-21,Orthopedics,Completed
4,AP005,HP005,DR02,2024-04-22,Neurology,Cancelled


In [50]:
# ---------- Clean lab reports ----------
# Problem: one test value is missing, and one is the text "not done" instead of a number.
# What we do: convert the column to numbers (anything invalid becomes missing),
#             then fill missing values with the median value for that same test.
# Why: a lab value needs to be a real number so we can average or compare it later.
#      Using the median per test (not one median for everything) is more accurate,
#      since Blood Pressure and Glucose have very different normal ranges.
lab_reports_clean = lab_reports.copy()
lab_reports_clean["test_value"] = pd.to_numeric(lab_reports_clean["test_value"], errors="coerce")
lab_reports_clean["test_value"] = lab_reports_clean.groupby("test_name")["test_value"].transform(
    lambda x: x.fillna(x.median())
)

# Problem: test dates are stored as plain text.
# What we do: convert to a real date type.
lab_reports_clean["test_date"] = pd.to_datetime(lab_reports_clean["test_date"], errors="coerce")

lab_reports_clean.head()


,lab_id,patient_id,test_name,test_value,test_unit,test_date
0,LB001,HP001,Blood Pressure,118.0,mmHg,2024-04-20
1,LB002,HP002,Vitamin D,22.5,ng/mL,2024-04-20
2,LB003,HP003,Thyroid Level,3.1,uIU/mL,2024-04-21
3,LB004,HP004,Glucose,96.0,mg/dL,2024-04-21
4,LB005,HP005,Blood Pressure,122.5,mmHg,2024-04-22


In [51]:
# ---------- Clean wearable data ----------
# Problem: one heart rate reading is missing, and one step count is negative
#          (a person cannot walk a negative number of steps - it's a device error).
# What we do: convert both columns to numbers, treat negative steps as missing,
#             then fill missing heart rate / steps with the median of that column.
wearable_clean = wearable_data.copy()
wearable_clean["heart_rate"] = pd.to_numeric(wearable_clean["heart_rate"], errors="coerce")
wearable_clean["steps"] = pd.to_numeric(wearable_clean["steps"], errors="coerce")
wearable_clean.loc[wearable_clean["steps"] < 0, "steps"] = pd.NA
wearable_clean["steps"] = pd.to_numeric(wearable_clean["steps"], errors="coerce")

wearable_clean["heart_rate"] = wearable_clean["heart_rate"].fillna(wearable_clean["heart_rate"].median()).astype(int)
wearable_clean["steps"] = wearable_clean["steps"].fillna(wearable_clean["steps"].median()).astype(int)

# Problem: sleep hours is stored as plain text/numbers mixed with the rest of the column.
# What we do: make sure it's numeric.
wearable_clean["sleep_hours"] = pd.to_numeric(wearable_clean["sleep_hours"], errors="coerce")
wearable_clean["sleep_hours"] = wearable_clean["sleep_hours"].fillna(wearable_clean["sleep_hours"].median())

wearable_clean["record_date"] = pd.to_datetime(wearable_clean["record_date"], errors="coerce")

wearable_clean.head()


,record_id,patient_id,record_date,heart_rate,steps,sleep_hours
0,WR001,HP001,2024-04-20,74,6100,7.1
1,WR002,HP002,2024-04-20,81,5300,6.4
2,WR003,HP003,2024-04-21,77,7200,7.6
3,WR004,HP004,2024-04-21,76,5400,6.2
4,WR005,HP005,2024-04-22,67,3100,8.1


In [52]:
# ---------- Clean doctor notes ----------
# Problem: an exact duplicate note row.
# What we do: drop duplicate rows.
notes_clean = doctor_notes.drop_duplicates().copy()

# Problem: one note is missing a diagnosis.
# What we do: fill it with "Not specified" instead of leaving it blank.
# Why: we can't guess a medical diagnosis, so we label it honestly instead.
notes_clean["diagnosis"] = notes_clean["diagnosis"].fillna("Not specified").replace("", "Not specified")
notes_clean["notes"] = notes_clean["notes"].fillna("No additional notes")

notes_clean["visit_date"] = pd.to_datetime(notes_clean["visit_date"], errors="coerce")

notes_clean.head()


,note_id,patient_id,doctor_id,diagnosis,notes,visit_date
0,DN001,HP001,DR01,Mild Hypertension,Advised regular BP monitoring,2024-04-20
1,DN002,HP002,DR02,Migraine,Prescribed pain relief and rest,2024-04-20
2,DN003,HP003,DR03,Not specified,Patient reported fatigue and low appetite,2024-04-21
3,DN004,HP004,DR04,Knee Strain,Recommended physiotherapy,2024-04-21
4,DN005,HP005,DR02,Tension Headache,Suggested stress management techniques,2024-04-22


In [53]:
print("=" * 40)
print("         TRANSFORM STEP")
print("=" * 40)
print(f"Patients       : {len(patients)} → {len(patients_clean)} rows")
print(f"Appointments   : {len(appointments)} → {len(appointments_clean)} rows")
print(f"Lab Reports    : {len(lab_reports)} → {len(lab_reports_clean)} rows")
print(f"Wearable Data  : {len(wearable_data)} → {len(wearable_clean)} rows")
print(f"Doctor Notes   : {len(doctor_notes)} → {len(notes_clean)} rows")
print("=" * 40)


         TRANSFORM STEP
Patients       : 19 → 18 rows
Appointments   : 19 → 18 rows
Lab Reports    : 18 → 18 rows
Wearable Data  : 18 → 18 rows
Doctor Notes   : 19 → 18 rows


## ✅ Step 4: Validate

Validation is the final quality check. We make sure the cleaned data looks correct before putting it into the database.

We use simple `assert` statements: if a check is True, nothing happens and the notebook keeps going. If a check is False, Python stops immediately with an error - which is exactly what we want, because bad data should never quietly make it into MySQL.


In [54]:
def validate_patients(df):
    assert df["patient_id"].notna().all(), "Some patient_id values are missing"
    assert (df["age"] >= 0).all(), "Found a negative age"
    assert df["gender"].isin(["Male", "Female"]).all(), "Found an unexpected gender value"
    assert df["registration_date"].notna().all(), "Some registration dates are invalid"


def validate_appointments(df, patients_df):
    valid_ids = set(patients_df["patient_id"])
    assert df["appointment_id"].notna().all(), "Some appointment_id values are missing"
    assert df["patient_id"].isin(valid_ids).all(), "Found an appointment with an unknown patient_id"
    assert df["appointment_date"].notna().all(), "Some appointment dates are invalid"


def validate_lab_reports(df, patients_df):
    valid_ids = set(patients_df["patient_id"])
    assert df["lab_id"].notna().all(), "Some lab_id values are missing"
    assert df["patient_id"].isin(valid_ids).all(), "Found a lab report with an unknown patient_id"
    assert df["test_value"].notna().all(), "Some lab test values are still missing"


def validate_wearable_data(df):
    assert df["record_id"].notna().all(), "Some record_id values are missing"
    assert df["heart_rate"].between(30, 220).all(), "Found an unrealistic heart rate"
    assert (df["steps"] >= 0).all(), "Found negative steps"


def validate_doctor_notes(df, patients_df):
    valid_ids = set(patients_df["patient_id"])
    assert df["note_id"].notna().all(), "Some note_id values are missing"
    assert df["patient_id"].isin(valid_ids).all(), "Found a doctor note with an unknown patient_id"
    assert df["visit_date"].notna().all(), "Some visit dates are invalid"


In [55]:
print("=" * 40)
print("        VALIDATION RESULTS")
print("=" * 40)

validate_patients(patients_clean)
print("Patients       : \u2713 PASSED")

validate_appointments(appointments_clean, patients_clean)
print("Appointments   : \u2713 PASSED")

validate_lab_reports(lab_reports_clean, patients_clean)
print("Lab Reports    : \u2713 PASSED")

validate_wearable_data(wearable_clean)
print("Wearable Data  : \u2713 PASSED")

validate_doctor_notes(notes_clean, patients_clean)
print("Doctor Notes   : \u2713 PASSED")

print("=" * 40)
print("All validation checks passed!")
print("=" * 40)


        VALIDATION RESULTS
Patients       : ✓ PASSED
Appointments   : ✓ PASSED
Lab Reports    : ✓ PASSED
Wearable Data  : ✓ PASSED
Doctor Notes   : ✓ PASSED
All validation checks passed!


## 🗄️ Step 5: Load into MySQL

After the data is cleaned and validated, we store it in MySQL. This allows us to use SQL to search and analyze the hospital data, without needing to run any Python code.

We connect using SQLAlchemy's `create_engine()`, which opens a "pipe" between Python and MySQL using the connection details below.

**Never put a real password directly in a notebook.** Here we read the credentials from environment variables (or a `.env` file), with placeholder values shown only as an example of the *shape* of a real value - not real credentials.


In [56]:
import os
from urllib.parse import quote_plus

# It's best practice to load these from environment variables or a .env file
# (e.g. using python-dotenv) instead of typing them directly here.
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "Javeed@786")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "Varshahos")

DB_PASSWORD = quote_plus("Javeed@786")
CONNECTION_STRING = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("=" * 40)
print("       DATABASE CONNECTION")
print("=" * 40)
print(f"Database : {DB_NAME}")
print(f"Host     : {DB_HOST}")

engine = None
try:
    from sqlalchemy import create_engine, text

    engine = create_engine(CONNECTION_STRING)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Status   : \u2713 Connected Successfully")
except Exception as error:
    engine = None
    print("Status   : \u2717 Connection failed")
    print(f"Details  : {error}")
    print()
    print("Make sure MySQL is running, that 'Hospitaldb' exists")
    print("(CREATE DATABASE Hospitaldb;), that 'pip install sqlalchemy")
    print("mysql-connector-python' has been run, and that your DB_USER /")
    print("DB_PASSWORD environment variables are set correctly.")
print("=" * 40)


       DATABASE CONNECTION
Database : Varshahos
Host     : localhost
Status   : ✓ Connected Successfully


## 📦 Step 6: Load the Cleaned Data into MySQL

Now that we have a working connection, we write each cleaned DataFrame into its own MySQL table using pandas' `to_sql()`. `if_exists="replace"` means each time we run this notebook, MySQL gets a fresh, up-to-date copy of the cleaned data.


In [57]:
clean_tables = {
    "patients": patients_clean,
    "appointments": appointments_clean,
    "lab_reports": lab_reports_clean,
    "wearable_data": wearable_clean,
    "doctor_notes": notes_clean,
}

print("=" * 40)
print("             LOAD STEP")
print("=" * 40)

if engine is not None:
    try:
        for table_name, df in clean_tables.items():
            df.to_sql(table_name, con=engine, if_exists="replace", index=False)
            print(f"{table_name:<15} \u2192 {len(df)} rows loaded")
        print("=" * 40)
        print("All data successfully loaded into MySQL!")
    except Exception as error:
        print(f"Loading failed: {error}")
else:
    print("Skipped - no active database connection.")
    print("Run this cell again once Step 5 shows a successful connection.")
print("=" * 40)


             LOAD STEP
patients        → 18 rows loaded
appointments    → 18 rows loaded
lab_reports     → 18 rows loaded
wearable_data   → 18 rows loaded
doctor_notes    → 18 rows loaded
All data successfully loaded into MySQL!


## 🔎 Step 7: Verify the Data in MySQL

Finally, we run a couple of simple SQL queries against MySQL itself to confirm the cleaned data was actually stored. This is a sanity check - it confirms that the cleaned data was successfully stored in MySQL.


In [58]:
if engine is not None:
    try:
        tables = pd.read_sql("SHOW TABLES;", con=engine)
        display(tables)
    except Exception as error:
        print(f"Could not list tables: {error}")
else:
    print("No active database connection - skipping verification.")


,Tables_in_varshahos
0,appointments
1,doctor_notes
2,lab_reports
3,patients
4,wearable_data


In [59]:
if engine is not None:
    try:
        preview = pd.read_sql("SELECT * FROM patients LIMIT 5;", con=engine)
        display(preview)
    except Exception as error:
        print(f"Could not preview 'patients' table: {error}")
else:
    print("No active database connection - skipping verification.")


,patient_id,name,age,gender,phone,registration_date
0,HP001,Nikhil Bhat,29,Male,9123456780,2024-04-01
1,HP002,Ayesha Khan,34,Female,9123456781,2024-04-02
2,HP003,Rohan Deshmukh,41,Male,9123456782,2024-04-03
3,HP004,Kavya Suresh,26,Female,9123456783,2024-04-04
4,HP005,Manoj Tiwari,50,Male,9123456784,2024-04-04


## 📊 Step 8: SQL Analytics

Now that our clean data lives in MySQL, we can explore it using plain SQL. Below are 10 simple, beginner-friendly analytical queries. Each one is explained in a sentence or two - and since this is fictional demo data, none of these results represent real medical findings.


In [60]:
def run_query(sql, description):
    print(f"📌 {description}")
    if engine is None:
        print("   (skipped - no active database connection)\n")
        return
    try:
        result = pd.read_sql(sql, con=engine)
        display(result)
    except Exception as error:
        print(f"   Query failed: {error}")
    print()


## 🎯 Final Pipeline Result


In [61]:
status = "\u2713 Connected" if engine is not None else "\u2717 Not connected"

print("=" * 45)
print("     HOSPITAL DATA PIPELINE COMPLETED")
print("=" * 45)
print()
print("📥 Extract      \u2713 Completed")
print("🧹 Transform    \u2713 Completed")
print("✅ Validate     \u2713 Passed")
print(f"🗄️ Load         {status}")
print("📊 Analytics    \u2713 Completed")
print()
print(f"Database: {DB_NAME}")
print()
print("Tables:")
print("\u2713 patients")
print("\u2713 appointments")
print("\u2713 lab_reports")
print("\u2713 wearable_data")
print("\u2713 doctor_notes")
print()
print("=" * 45)
print("     PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 45)


     HOSPITAL DATA PIPELINE COMPLETED

📥 Extract      ✓ Completed
🧹 Transform    ✓ Completed
✅ Validate     ✓ Passed
🗄️ Load         ✓ Connected
📊 Analytics    ✓ Completed

Database: Varshahos

Tables:
✓ patients
✓ appointments
✓ lab_reports
✓ wearable_data
✓ doctor_notes

     PIPELINE COMPLETED SUCCESSFULLY


## 🏥 How This Solves the Hospital Case Study

```text
Hospital Data Sources
        ↓
      Extract
        ↓
      Clean
        ↓
     Validate
        ↓
      MySQL
        ↓
   SQL Analytics
        ↓
Future ML / Risk Prediction
```

The hospital's data is scattered across registration, appointments, the lab, wearable devices, and doctor notes. This notebook brings all of that data together, cleans it, checks it, and stores it in one organized place (MySQL). Once the data is organized like this, it's much easier to analyze - and it's also in the right shape to be used by a Machine Learning model later, if the hospital wants to predict which patients might be at higher risk.

**Data Engineering prepares the data. Machine Learning would use that prepared data later.** This notebook only does the first part, on purpose.


## 🚀 Future Scope

This notebook covers the basics of Data Engineering. A few realistic next steps:

- Collecting data from more hospital departments
- Building a dashboard to visualize the data
- Automating data collection instead of using static CSV files
- Training a Machine Learning model to predict high-risk patients
- Deeper appointment and scheduling analytics
